# Decision Trees and Random Forests

In this notebook, we are going to use decision trees and random forest algorithms to improve our predictions on student grades based on our known features. 

FYI, the dataset is synthetic, I do not have any mean to monitor the time you spend studying my course.

## Importing the dependencies

First, we are going to import all the dependencies that we will need for this lab. If you cannot run the following code cell, do not forget to [create an environment](https://docs.astral.sh/uv/guides/projects/), to install the dependencies inside of it (using the command `uv add -r requirements.txt`) and to use it as your Jupyter kernel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("./synthetic_student_data.csv")
df.head()

## Example with a single input feature

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df["hours_studied"], df["grade"], c="skyblue", marker="+")

ax.set_xlabel("Number of hours studied", fontsize="large")
ax.set_ylabel("Grade", fontsize="large")
ax.set_ylim((0,20))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout()

Let's now try to train a decision tree with different `max_depth` values. The goal here is to find the value which allows us to describe the data as well as possible without overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

X = df["hours_studied"].to_numpy().reshape(-1, 1)
Y = df["grade"].to_numpy()

ax.scatter(X, Y, c="skyblue", marker="+")

ax.set_xlabel("Number of hours studied", fontsize="large")
ax.set_ylabel("Grade", fontsize="large")
ax.set_ylim((0,20))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

model_d2 = DecisionTreeRegressor(max_depth=2)
model_d2.fit(X, Y)

model_d3 = DecisionTreeRegressor(max_depth=3)
model_d3.fit(X, Y)

model_d5 = DecisionTreeRegressor(max_depth=5)
model_d5.fit(X, Y)

# Predict line over full x range
x_vals = np.linspace(df["hours_studied"].min(), df["hours_studied"].max(), 100).reshape(-1, 1)
y_vals_d2 = model_d2.predict(x_vals)
y_vals_d3 = model_d3.predict(x_vals)
y_vals_d5 = model_d5.predict(x_vals)

ax.plot(x_vals, y_vals_d2, c="r", label="max_depth=2")
ax.plot(x_vals, y_vals_d3, c="g", label="max_depth=3")
ax.plot(x_vals, y_vals_d5, c="violet", label="max_depth=5")
ax.legend()
fig.tight_layout()

What do you think would be the correct `max_depth` value to pick? Why?

## Improved prediction with multiple variables

Here, we will take into account the multiple parameters in our dataset. The results would be much more complex to plot so we will simply display the decision tree itself.

To facilitate interpretation, we will not standardize the features here (decision trees and random forests are some of the rare cases which do not actuallly need it). However, in general case, do not forget to at least ask yourself "Should I standardize my features?" and if you're not sure about the answer, standardize them anyway.

In [ ]:
input_features = ["hours_studied", "sleep_hours", "class_attendance"]
X = df[input_features].to_numpy()
Y = df["grade"].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=4321)

model = DecisionTreeRegressor(max_depth=3)
model.fit(X_train, Y_train)

fig, ax = plt.subplots(figsize=(15, 8))
plot_tree(model, feature_names=input_features, proportion=True, filled=True, ax=ax)
fig.tight_layout()

score = model.score(X_test, Y_test)
print(f"R² score = {score:.3f}")

## From a single (Decision) Tree to a (Random) Forest

As you can see, the results with a single decision tree are not so convincing compared with our previous polynomial regression. Let's now try to combine the predictions of multiple decision trees to improve the results.

In [ ]:
input_features = ["hours_studied", "sleep_hours", "class_attendance"]
X = df[input_features].to_numpy()
Y = df["grade"].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=4321)

model = RandomForestRegressor(n_estimators=300, random_state=4321)  # Random Forest reduces the risk of overfitting so we do not need to set a maximum tree depth
model.fit(X_train, Y_train)

# fig, ax = plt.subplots(figsize=(15, 8))
# plot_tree(model, feature_names=input_features, proportion=True, ax=ax)

score = model.score(X_test, Y_test)
print(f"R² score = {score:.3f}")

We can now take a look at individual decision trees composing our random forest:

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
plot_tree(model.estimators_[0], feature_names=input_features, proportion=True, filled=True, ax=ax)
fig.tight_layout()

In [ ]:
importances = pd.Series(model.feature_importances_, index=input_features)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(importances.index, importances.values, color="#4C72B0")
ax.set_xlabel("Importance", size="large")
plt.tight_layout()
plt.show()